In [1]:
import sys
import os
import pandas as pd
import numpy as np
import json
import re
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Add backend to path so we can import the system's actual modules
sys.path.append(os.path.abspath('../../'))

from backend.services.parser import parser
from backend.services.embedding import embedding_service
from backend.services.scorer import scorer

# Load dataset
path = "./data/UpdatedResumeDataSet.csv"
df = pd.read_csv(path)

# Quick text cleaner for the resumes
def clean_text(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    text = url_pattern.sub(r'', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text, flags=re.IGNORECASE)
    text = " ".join(text.split())
    return text

df['Resume'] = df['Resume'].apply(clean_text)
print(f"Loaded {len(df)} resumes.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 962 resumes.


## 1. Generate Job Requirements Context
Instead of generating a text job description and then using a resume parser on it, we will use a single prompt to generate BOTH a gold-standard job description AND a structured list of required skills for each category.

In [2]:
from google import genai
from google.genai import types

# The backend parser already initialized the Gemini client with the API key
client = parser.client

def generate_job_context(category: str) -> dict:
    prompt = f"""
    You are an expert HR recruiter. We need a gold-standard job description and a list of REQUIRED technical skills for the following job category: '{category}'.
    
    Return ONLY valid JSON matching this schema exactly:
    {{
        "job_description": "A comprehensive 2-3 paragraph job description.",
        "required_skills": ["skill1", "skill2", "skill3"]
    }}
    
    Make sure to use lowercase canonical names for skills (e.g. "python", "react"). Do not include any markdown block formatting like ```json in the response, just the raw JSON text.
    """
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.0
        ),
    )
    
    try:
        return json.loads(response.text)
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON for category: {category}")
        return {"job_description": "", "required_skills": []}

# Generate context dictionary
job_requirements = {}
categories = df['Category'].unique()

print("Generating Job Context for all categories...")
for category in tqdm(categories, desc="Categories"):
    context = generate_job_context(category)
    
    jd_text = context.get('job_description', '')
    req_skills = context.get('required_skills', [])
    
    # Generate embeddings
    try:
        jd_embedding = embedding_service.generate_embedding(jd_text) if jd_text else []
        skills_text = " ".join(req_skills)
        skill_embedding = embedding_service.generate_embedding(skills_text) if skills_text else jd_embedding
    except Exception as e:
        print(f"Embedding failed for {category}: {e}")
        jd_embedding, skill_embedding = [], []

    job_requirements[category] = {
        'job_description': jd_text,
        'required_skills': req_skills,
        'job_embedding': jd_embedding,
        'skills_vector': skill_embedding
    }

print("Finished generating Job Requirements Context.")

Generating Job Context for all categories...


Categories: 100%|██████████| 25/25 [02:25<00:00,  5.83s/it]

Finished generating Job Requirements Context.


## 2. Evaluate Candidates
Run each candidate through the actual parser to extract skills and text embeddings, then test against all job category requirements.

In [3]:
# We use a subset for quick testing (2 per category). Remove `.head(2)` to evaluate the whole dataset.
sample_df = df.groupby('Category').head(2).reset_index(drop=True)

results = []
true_labels = []
predicted_labels = []

# Safe parser wrapper handling the backend Gemini parser
def safe_parse_skills(text):
    try:
        llm_data = parser._extract_with_gemini(text)
        return llm_data.get('skills', [])
    except Exception as e:
        return []

for index, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Scoring Resumes"):
    true_cat = row['Category']
    resume_text = row['Resume']
    
    # 1. Candidate Parsing
    candidate_skills = safe_parse_skills(resume_text)
    
    # 2. Candidate Embeddings
    try:
        resume_vector = embedding_service.generate_embedding(resume_text)
        skills_str = " ".join(candidate_skills)
        if skills_str:
            candidate_skills_vector = embedding_service.generate_embedding(skills_str)
        else:
            candidate_skills_vector = resume_vector # fallback
    except Exception as e:
        print(f"Embedding error on index {index}: {e}")
        continue
        
    # 3. Scoring
    best_category = None
    best_score = -1
    
    for category, reqs in job_requirements.items():
        try:
            scores = scorer.calculate_match(
                job_embedding=reqs['job_embedding'],
                candidate_embedding=resume_vector,
                job_requirements_embedding=reqs['skills_vector'],
                candidate_skills_embedding=candidate_skills_vector,
                job_skills=reqs['required_skills'],
                candidate_text=resume_text,
                candidate_skills=candidate_skills
            )
            total = scores['total_score']
            if total > best_score:
                best_score = total
                best_category = category
        except Exception as e:
            pass # Skip if scoring fails
            
    if best_category:
        true_labels.append(true_cat)
        predicted_labels.append(best_category)
        results.append({
            'true_category': true_cat,
            'predicted_category': best_category,
            'score': best_score
        })


Scoring Resumes: 100%|██████████| 50/50 [11:03<00:00, 13.28s/it]


In [4]:
if true_labels:
    acc = accuracy_score(true_labels, predicted_labels)
    prec = precision_score(true_labels, predicted_labels, average='weighted', zero_division=0)
    rec = recall_score(true_labels, predicted_labels, average='weighted', zero_division=0)
    f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=0)

    print(f"Evaluation Complete! Processed {len(true_labels)} resumes.")
    print("-" * 30)
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(true_labels, predicted_labels, zero_division=0))
else:
    print("No results processed.")

Evaluation Complete! Processed 50 resumes.
------------------------------
Accuracy:  0.6800
Precision: 0.6593
Recall:    0.6800
F1-Score:  0.6349

Detailed Classification Report:
                           precision    recall  f1-score   support

                 Advocate       0.00      0.00      0.00         2
                     Arts       0.00      0.00      0.00         2
       Automation Testing       0.00      0.00      0.00         2
               Blockchain       1.00      1.00      1.00         2
         Business Analyst       0.40      1.00      0.57         2
           Civil Engineer       1.00      0.50      0.67         2
             Data Science       1.00      1.00      1.00         2
                 Database       1.00      1.00      1.00         2
          DevOps Engineer       1.00      0.50      0.67         2
         DotNet Developer       0.50      1.00      0.67         2
            ETL Developer       0.50      0.50      0.50         2
   Electrical En